# ?? Modal Logic Mechanistic Interpretability: Google Colab Runner
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

Welcome to the **Modal Logic Mechanistic Interpretability** interactive cloud execution environment.
This notebook allows you to run full circuit discovery (Part A), macroscopic staged analysis (Part B), and comparative baseline evaluations on Transformer-based Large Language Models (such as `Qwen/Qwen3.5-2B`, `Qwen/Qwen3.5-4B`, and `Qwen/Qwen3.5-9B`) using GPU cloud compute.

---
### ?? Pipeline Summary:
1. **Part A (Circuit Discovery)**: Controlled Mediation Analysis (CMA) activation patching, discovering 7 functional attention head families (`MOH`, `MPH`, `CRH`, `QRLH`, `QRMH`, `FPH`, `DH`), sufficiency ablation tables, and circuit diagrams.
2. **Part B (Macroscopic Mechanisms)**: 4-Region MLP Staging (Facts, Accessibility, Expression, Query), Residual stream transmission, Fact Retrospection contrast across accessible/inaccessible worlds, and 11 modal rule/axiom categories (including Modal Axioms **B, D, 4, 5, K, T**).
3. **Comparative Evaluation**: Side-by-side benchmark comparing First-Order Propositional Logic vs. Modal Logic.

## 1. ?? Environment Setup & Hardware Inspection
Let's configure paths, verify GPU availability (CUDA), and optionally mount Google Drive.

In [ ]:
#@title Check Hardware & Environment
import os
import sys
from pathlib import Path

# Clone repo if running fresh in Colab
if not Path("modal-logic-mi").exists() and not Path("../modal-logic-mi").exists():
    !git clone https://github.com/artemiui/tblm-modal-reasoning.git /content/tblm-modal-reasoning
    %cd /content/tblm-modal-reasoning

# Add project root and subdirectories to sys.path
root_dir = Path.cwd() if Path("modal-logic-mi").exists() else Path.cwd().parent
sys.path.insert(0, str(root_dir))
sys.path.insert(0, str(root_dir / "modal-logic-mi"))
sys.path.insert(0, str(root_dir / "modal-logic-transformer-circuit"))
sys.path.insert(0, str(root_dir / "colab"))

from colab_utils import setup_colab_environment, print_gpu_info, login_huggingface, display_image, export_results_zip

mount_google_drive = False #@param {type:"boolean"}
setup_colab_environment(mount_drive=mount_google_drive)

## 2. ?? Install Dependencies
Install optimized dependencies (`transformer_lens`, `torch`, `transformers`, `accelerate`, `matplotlib`, `pandas`, `pyyaml`).

In [ ]:
#@title Install Required Packages
!pip install -q -r colab/requirements-colab.txt
print("? All dependencies installed successfully!")

## 3. ?? Hugging Face Authentication (Optional)
If using gated model checkpoints or downloading directly from Hugging Face Hub, you can supply your HF token below:

In [ ]:
#@title Hugging Face Access Token
hf_token = "" #@param {type:"string"}
login_huggingface(hf_token)

## 4. ??? Interactive Pipeline Execution
Select your model preset and pipeline stage. You can run pre-flight unit tests, Part A circuit discovery, Part B macroscopic staging, or the full pipeline.

In [ ]:
#@title Configure & Run Experiment
#@markdown ### Choose Model and Stage:
model_preset = "Qwen/Qwen3.5-2B" #@param ["Qwen/Qwen3.5-2B", "Qwen/Qwen3.5-4B", "Qwen/Qwen3.5-9B", "google/gemma-2-9b-it", "mistralai/Mistral-7B-v0.1"]
pipeline_stage = "all" #@param ["all", "part_a", "part_b", "comparative", "unit_tests"]
device = "cuda" #@param ["cuda", "cpu"]
run_full_weights = True #@param {type:"boolean"}

from scripts.run_project import PreflightCheckpoints, ProjectExecutor

print(f"\n?? Starting execution for {model_preset} on {device} (Stage: {pipeline_stage})...\n")

# Run Pre-flight Checks
preflight = PreflightCheckpoints(model_id=model_preset, device=device)
preflight.run_all_checkpoints(root_dir / "modal-logic-mi")

# Execute Project Pipeline
executor = ProjectExecutor(
    repo_root=root_dir / "modal-logic-mi",
    model_id=model_preset,
    device=device,
    run_full=run_full_weights
)

if pipeline_stage in ["unit_tests", "all"]:
    test_results = executor.run_unit_tests()

if pipeline_stage in ["part_a", "all"]:
    part_a_results = executor.run_part_a_pipeline()

if pipeline_stage in ["part_b", "all"]:
    part_b_results = executor.run_part_b_pipeline()

if pipeline_stage in ["comparative", "all"]:
    comp_results = executor.run_comparative_baseline()

print("\n?? Pipeline execution completed successfully!")

## 5. ?? Inline Results & Visualizations
Inspect discovered circuit tables, sufficiency metrics, and publication figures directly inside the notebook.

In [ ]:
#@title Display Discovered Circuit Architecture & Tables
import pandas as pd
from pathlib import Path

# Display Circuit Diagram if generated
circuit_diagram_path = root_dir / "modal-logic-mi" / "results" / "project_runs" / "circuit_diagram.png"
if circuit_diagram_path.exists():
    print("?? Discovered Modal Logic Circuit Architecture:")
    display_image(circuit_diagram_path, width=800)

# Display Sufficiency Ablation Table
suff_table_path = root_dir / "modal-logic-mi" / "results" / "project_runs" / "test_sufficiency" / "sufficiency_table.csv"
if suff_table_path.exists():
    print("\n?? Circuit Sufficiency Ablation Matrix:")
    df = pd.read_csv(suff_table_path)
    display(df)

## 6. ?? Export Results
Create a downloadable zip archive or sync directly to Google Drive.

In [ ]:
#@title Export Results Archive
results_dir = root_dir / "modal-logic-mi" / "results"
zip_path = export_results_zip(results_dir, "modal_logic_experiment_results.zip", trigger_download=True)